### Utility Functions

In [1]:
# This part helps with stability on multi-GPU systems with great inbalance between the GPUs (eg. integrated vs discrete gpu)
# IMPORTANT must be done before importing torch, else session must be restarted
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch

for i in range(torch.cuda.device_count()):
    free = torch.cuda.mem_get_info(i)[0]
    total = torch.cuda.mem_get_info(i)[1]

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free:  {free / 1024**3:.2f} GB")
    print(f"  Total: {total / 1024**3:.2f} GB")

GPU 0: AMD Radeon RX 6700S
  Free:  7.96 GB
  Total: 7.98 GB


In [ ]:
from util import clear_folder

# Clears results of the last run, not necessary, just to reduce folder size

clear_folder("./results")

In [ ]:
from util import clear_cuda_cache

# If model gets stuck during training, uncomment the following line

clear_cuda_cache()

## Load Dataset

This portion loads dataset, and assigns id for each label

#### English

This loads english version of the dataset

In [2]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-combined",
    split="train",
    label2id=label2id
)


Loaded datasets (train=44323, val=5540, test=5541)


#### Slovenian

This loads slovenian version of the dataset

In [ ]:
from util import load_datasets_from_hf

label2id={"ham": 0, "spam": 1}

train_dataset, val_dataset, test_dataset, id2label = load_datasets_from_hf(
    dataset_name="daviiiidcpp1/sms-spam-slovene",
    split="train",
    label2id=label2id
)


## Model settings 

### XLM-RoBERTA

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "FacebookAI/xlm-roberta-base"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/47392 [00:00<?, ? examples/s]

Map:   0%|          | 0/5923 [00:00<?, ? examples/s]

Map:   0%|          | 0/5926 [00:00<?, ? examples/s]

### TinyBert

In [ ]:
from util import tokenize_datasets, load_model_and_tokenizer, build_trainer


model_name = "huawei-noah/TinyBERT_General_4L_312D"

model, tokenizer = load_model_and_tokenizer(model_name, num_labels=2)

train_dataset, val_dataset, test_dataset = tokenize_datasets(train_dataset, val_dataset, test_dataset, tokenizer)

trainer = build_trainer(model, train_dataset, val_dataset)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Map:   0%|          | 0/44323 [00:00<?, ? examples/s]

Map:   0%|          | 0/5540 [00:00<?, ? examples/s]

Map:   0%|          | 0/5541 [00:00<?, ? examples/s]

## Train

In [ ]:
from datetime import datetime
from util import clear_folder

clear_folder("./results")

trainer.train()

model_save_folder = "trained_models" 

model_save_name = model_name.split("/")[-1] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
trainer.save_model(os.path.join(model_save_folder, model_save_name))
tokenizer.save_pretrained(os.path.join(model_save_folder, model_save_name))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.114337,0.156529,0.969314,0.940222,0.980193,0.959792
2,0.144070,0.147560,0.976354,0.955806,0.982126,0.968787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.073727,0.127173,3,0.979065,0.977002,0.972211,0.974601


{'eval_loss': 0.1271727830171585,
 'eval_accuracy': 0.9790646631774439,
 'eval_precision': 0.9770020533880903,
 'eval_recall': 0.9722108704536166,
 'eval_f1': 0.9746005735354363}

### Data Visualization

In [ ]:
import graphs
import importlib

importlib.reload(graphs)

graphs.generate_all_plots("./results", "./graphs" + "/" + model_name.split("/")[-1])